# 1. Which TPU did Colab give us?

In [8]:
# (v5e-1 = 16 GB HBM, v6e-1 = 32 GB HBM)
!tpu-info


Libtpu version: 0.0.32.1                                                        
Accelerator type: v5e                                                           
                                                                                
TPU Chips                                      
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━┓
┃ Chip        ┃ Type         ┃ Devices ┃ PID  ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━┩
│ /dev/vfio/0 │ TPU v5e chip │ 1       │ None │
└─────────────┴──────────────┴─────────┴──────┘
╭───────────────────────── Runtime Utilization Status ─────────────────────────╮
│ WARNING: Libtpu metrics unavailable. Is there a framework using the TPU? See │
│ ]8;id=11085121;https://github.com/google/cloud-accelerator-diagnostics/tree/main/tpu_info\tpu_info docs]8;;\ for more information.                                          │
╰──────────────────────────────────────────────────────────────────────────────╯
TPU Runtime Utilization                
┏━━━━━

# 2. Install SGLang's TPU backend.

In [2]:
!pip install -U "sglang-jax[tpu]"

!python -c "from sgl_jax import check_env; check_env.check_env()"


/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
sglang-jax: 0.0.2
jax: 0.8.3
jaxlib: 0.8.3
triton: Module Not Found
transformers: 4.55.4
numpy: 2.2.6
aiohttp: 3.14.3
fastapi: 0.116.2
huggingface_hub: 0.34.6
modelscope: 1.28.2
orjson: 3.11.9
packaging: 26.3
psutil: 7.0.0
pydantic: 2.11.10
python-multipart: 0.0.32
pyzmq: 27.0.2
uvicorn: 0.35.0
uvloop: 0.21.0
openai: 3.6.0
tiktoken: 0.14.0
[TPU v5 lite-0]: TPU_0(process=0,(0,0,0,0))
ulimit soft: 1048576


In [9]:
!pip install -q openai

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

cloudflared version 2026.8.3 (built 2026-08-31-10:04 UTC)


In [10]:
import importlib.util, pathlib

root = pathlib.Path(importlib.util.find_spec("sgl_jax").submodule_search_locations[0])
targets = [
    root / "srt/layers/attention/flash_attn_kernel/util.py",
    root / "srt/mem_cache/memory_pool.py",
]
OLD = "dtypes.bit_width("
NEW = 'getattr(dtypes, "bit_width", getattr(dtypes, "itemsize_bits", None))('
for f in targets:
    src = f.read_text()
    if OLD in src:
        f.write_text(src.replace(OLD, NEW))
        print("patched:", f.relative_to(root))
    elif NEW in src:
        print("already patched:", f.relative_to(root))
    else:
        print("call site not found (different sgl_jax version?):", f.relative_to(root))

already patched: srt/layers/attention/flash_attn_kernel/util.py
already patched: srt/mem_cache/memory_pool.py


# 3. Settings

In [11]:
MODEL = "Qwen/Qwen3-4B"   # ~8 GB in bf16
PORT  = 30000
LOG   = "/content/server.log"


# 4. Launch the server in the background

In [12]:
import os, sys, subprocess

env = dict(os.environ,
           JAX_COMPILATION_CACHE_DIR="/content/jit_cache",
           FLAX_ALWAYS_SHARD_VARIABLE="false")               # Flax >=0.12 eager sharding breaks sgl_jax model init
cmd = [
    sys.executable, "-u", "-m", "sgl_jax.launch_server",
    "--model-path", MODEL,
    "--trust-remote-code",
    "--device=tpu",
    "--tp-size=1",                  # one chip on Colab
    "--dtype=bfloat16",
    "--mem-fraction-static=0.8",    # drop to 0.6 on OOM
    "--attention-backend=fa",       # if the fa kernel fails on v5e, try: native
    "--chunked-prefill-size=2048",
    "--host=0.0.0.0",
    f"--port={PORT}",
]
server = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env)
print(f"started pid {server.pid}; logs -> {LOG}")

started pid 23789; logs -> /content/server.log


# 5. Wait for the server/health answers

In [13]:
import time, requests

WAIT_MINUTES = 40
POLL_SECONDS = 20

def server_is_up():
    try:
        return requests.get(f"http://localhost:{PORT}/health", timeout=5).ok
    except requests.RequestException:
        return False

def last_log_line():
    try:
        lines = [l for l in open(LOG).read().splitlines() if l.strip()]
        return lines[-1][:110] if lines else "(log is empty so far)"
    except FileNotFoundError:
        return "(no log yet)"

start = time.time()
for _ in range(WAIT_MINUTES * 60 // POLL_SECONDS):
    if server_is_up():
        break
    if server.poll() is not None:
        print(open(LOG).read()[-4000:])
        raise RuntimeError(f"server exited with code {server.returncode}")
    print(f"[{(time.time() - start) / 60:4.1f} min] not ready yet | {last_log_line()}")
    time.sleep(POLL_SECONDS)

if server_is_up():
    print(f"server is up ({(time.time() - start) / 60:.1f} min) -> run the next cell")
else:
    raise TimeoutError(f"still not up after {WAIT_MINUTES} min - check `!tail -n 40 {LOG}`")

[ 0.0 min] not ready yet | (log is empty so far)
[ 0.3 min] not ready yet | [LOADING] MODEL WEIGHTS:  33%|███▎      | 1/3 [00:06<00:13,  6.54s/file, file=model-00002-of-00003.safetensors
[ 0.7 min] not ready yet | [DECODE] PRECOMPILE:   0%|          | 0/3 [00:00<?, ?it/s, bs=1]
server is up (1.0 min) -> run the next cell


# 6. Talk to it with the OpenAI client

In [14]:
import openai

client = openai.OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="x")
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain TPUs in one paragraph"}],
    max_tokens=256,
)
print(resp.choices[0].message.content)


<think>
Okay, the user wants a one-paragraph explanation of TPUs. First, I need to recall what TPUs are. They are specialized hardware developed by Google for machine learning. I should mention their purpose: accelerating machine learning tasks, especially deep learning. Maybe start with their main function. Then, explain that they are designed to handle the computational demands of training and inference. Compare them to GPUs, maybe note that they are more efficient for certain tasks. Mention their use in Google's cloud services like Google Cloud AI Platform. Also, highlight their role in improving performance and reducing energy consumption. Need to keep it concise, so avoid too much technical jargon. Make sure to cover key points: purpose, design, efficiency, application in cloud services, and benefits like speed and energy savings. Check if there's anything else important. Oh, maybe mention that they are used in both training models and running inference. Alright, structure the par

# 7. Peek at the server log any time

In [15]:
!tail -n 40 {LOG}


[LOADING] MODEL WEIGHTS: 100%|██████████| 3/3 [00:13<00:00,  4.40s/file, file=model-00003-of-00003.safetensors]
[2026-09-01 20:30:06] Qwen3 weights loaded successfully!
[2026-09-01 20:30:06] ModelRunner kv_cache_dtype: <class 'jax.numpy.bfloat16'>
[2026-09-01 20:30:06] TPU Memory profiling: available_device_memory=8.3GB, available_kv_cache=5.1GB, max_tokens=37180, cell_size=147456bytes
[2026-09-01 20:30:06] ModelRunner max_total_num_tokens: 37180
[2026-09-01 20:30:06] Creating fused KV buffers for 36 layers
[2026-09-01 20:30:06] Total fused KV cache memory per layer: 0.14 GB, dtype: <class 'jax.numpy.bfloat16'>
[2026-09-01 20:30:06] Total time to create 36 buffers: 0.26 seconds
[2026-09-01 20:30:06] JAX Fused KV Cache allocated. #tokens: 37180, Fused KV size: 5.11 GB
[2026-09-01 20:30:06] Max running requests constraints:
[2026-09-01 20:30:06]   - Server limit: 18590 (max_total_tokens//2)
[2026-09-01 20:30:06]   - Token pool size: 2049
[2026-09-01 20:30:06]   - Attention backend: 3 (co

# 8. Open a public URL

In [16]:
import re, time, subprocess

TUNNEL_LOG = "/content/cloudflared.log"
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=open(TUNNEL_LOG, "w"), stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
for _ in range(30):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
    if m:
        PUBLIC_URL = m.group(0)
        break
    time.sleep(2)

if not PUBLIC_URL:
    tunnel.terminate()
    raise RuntimeError(f"no tunnel URL appeared - check {TUNNEL_LOG}")
print("public base URL :", PUBLIC_URL)
print("OpenAI endpoint :", PUBLIC_URL + "/v1")


public base URL : https://stored-interstate-trainers-enhancing.trycloudflare.com
OpenAI endpoint : https://stored-interstate-trainers-enhancing.trycloudflare.com/v1


# 11. Round-trip through the public URL

In [17]:
import time, requests

for _ in range(15):
    try:
        if requests.get(f"{PUBLIC_URL}/health", timeout=5).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)

r = requests.post(
    f"{PUBLIC_URL}/v1/chat/completions",
    json={"model": MODEL,
          "messages": [{"role": "user", "content": "Say hi through the tunnel in one line."}],
          "max_tokens": 64},
    timeout=600,
)
print(r.json()["choices"][0]["message"]["content"])

print("\n--- from any other machine ---")
print(f"""curl {PUBLIC_URL}/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -d '{{"model": "{MODEL}", "messages": [{{"role":"user","content":"hello"}}]}}'""")

<think>
Okay, the user wants me to say "hi through the tunnel in one line." Let me parse that. They probably want a greeting that's concise, maybe a metaphor or a creative phrase that involves a tunnel. Since it's one line, I need to make sure it's a single sentence.

Hmm,

--- from any other machine ---
curl https://stored-interstate-trainers-enhancing.trycloudflare.com/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "Qwen/Qwen3-4B", "messages": [{"role":"user","content":"hello"}]}'


In [18]:
# !curl https://stored-interstate-trainers-enhancing.trycloudflare.com/v1/chat/completions \
#   -H "Content-Type: application/json" \
#   -d '{"model": "Qwen/Qwen3-4B", "messages": [{"role":"user","content":"hello"}]}'

{"id":"5698111c11b04444a605792b92adb237","object":"chat.completion","created":1788294819,"model":"Qwen/Qwen3-4B","choices":[{"index":0,"message":{"role":"assistant","content":"<think>\nOkay, the user said \"hello\". I need to respond appropriately. Since they're just greeting me, I should reply in a friendly and welcoming manner. Maybe start with a greeting back, like \"Hi there!\" to keep it casual. Then, I can ask how I can assist them today. That way, they know I'm ready to help with whatever they need. I should keep it simple and not overcomplicate things. Let me make sure the tone is positive and open. Alright, that should work.\n</think>\n\nHi there! How can I assist you today? 😊","reasoning_content":null,"tool_calls":null},"logprobs":null,"finish_reason":"stop","matched_stop":151645,"hidden_states":null}],"usage":{"prompt_tokens":9,"total_tokens":130,"completion_tokens":121,"prompt_tokens_details":null}}

# 12. Close the public URL


In [ ]:
tunnel.terminate()

# 13. Stop the server

In [7]:
server.terminate()
try:
    server.wait(timeout=30)
except subprocess.TimeoutExpired:
    server.kill()

!pkill -f sgl_jax || true


^C
